# 06 Model specification
**Chapter 3: Methodology — B4 Model specification, worked example.** The Python twin of `06_model_specification.do`.

One research question per model, matching the RQ–method table in B2:

| RQ | Outcome | Level | Model |
|---|---|---|---|
| RQ1 Who uses e-wallets? | `ewallet_user` | binary | logit (LPM for comparison) |
| RQ2 How often do users use them? | `ewallet_days` 0–7 | count / ratio | OLS among users |
| RQ3 Who intends to keep using them? | `intention` 1–5 | ordinal | ordered logit |

In [1]:
# Setup: install anything missing (needed on Google Colab), then load the libraries.
import importlib.util, subprocess, sys
for pkg in ["pandas", "numpy", "scipy", "statsmodels", "linearmodels", "matplotlib", "openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# Data live in ../data next to this notebook; online (e.g. Colab) they are read from GitHub.
DATA = Path("../data")
RAW_URL = "https://raw.githubusercontent.com/soonjanjan-lang/Chapter3-Methodology-lectnotes-dashboard/main/downloads/data/"

def load(name):
    p = DATA / name
    return pd.read_csv(p) if p.exists() else pd.read_csv(RAW_URL + name)

In [2]:
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

df = load("ewallet_survey_clean.csv")
X = "PU + PS + EOU + finlit + female + ln_allow + urban"
df[["ewallet_user", "ewallet_days", "intention", "PU", "PS", "EOU", "finlit", "female", "ln_allow", "urban"]].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
ewallet_user,400.0,0.752,0.432,0.000,1.000,1.000,1.00,1.000
ewallet_days,393.0,2.318,2.126,0.000,0.000,2.000,4.00,7.000
intention,400.0,3.142,1.436,1.000,2.000,3.000,4.00,5.000
PU,400.0,3.143,1.012,1.000,2.500,3.250,4.00,5.000
PS,400.0,3.143,1.022,1.000,2.500,3.250,4.00,5.000
EOU,400.0,3.153,0.982,1.000,2.333,3.167,4.00,5.000
finlit,400.0,1.898,1.000,0.000,1.000,2.000,3.00,3.000
female,400.0,0.582,0.494,0.000,0.000,1.000,1.00,1.000
ln_allow,394.0,6.708,0.469,5.011,6.384,6.709,7.03,8.096
urban,400.0,0.635,0.482,0.000,0.000,1.000,1.00,1.000


## RQ1: binary outcome
$\Pr(user_i = 1 \mid x_i) = \Lambda(\beta_0 + \beta_1 PU_i + \dots + \beta_7 urban_i)$

In [3]:
lpm = smf.ols(f"ewallet_user ~ {X}", df).fit(cov_type="HC1")
logit = smf.logit(f"ewallet_user ~ {X}", df).fit(disp=False, cov_type="HC1")
print(pd.DataFrame({"LPM b": lpm.params, "logit b": logit.params, "logit se": logit.bse}).round(3))
print("N =", int(logit.nobs))
print(logit.get_margeff(at="overall").summary_frame().round(4))

           LPM b  logit b  logit se
Intercept -0.275   -5.722     2.137
PU         0.128    0.874     0.153
PS         0.088    0.618     0.148
EOU        0.046    0.320     0.150
finlit     0.010    0.092     0.139
female    -0.022   -0.118     0.285
ln_allow   0.029    0.222     0.300
urban      0.012   -0.019     0.281
N = 394
           dy/dx  Std. Err.       z  Pr(>|z|)  Conf. Int. Low  Cont. Int. Hi.
PU        0.1191     0.0189  6.3066    0.0000          0.0821          0.1561
PS        0.0842     0.0183  4.6008    0.0000          0.0483          0.1200
EOU       0.0436     0.0201  2.1712    0.0299          0.0042          0.0830
finlit    0.0125     0.0189  0.6614    0.5083         -0.0246          0.0496
female   -0.0160     0.0388 -0.4128    0.6798         -0.0921          0.0601
ln_allow  0.0302     0.0406  0.7437    0.4571         -0.0494          0.1099
urban    -0.0027     0.0384 -0.0692    0.9449         -0.0778          0.0725


## RQ2: days of use in the past week, users only

In [4]:
ols_days = smf.ols(f"ewallet_days ~ {X} + C(year_study)", df[df["ewallet_user"] == 1]).fit(cov_type="HC1")
print(ols_days.summary().tables[1])

                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -7.5474      1.416     -5.328      0.000     -10.324      -4.771
C(year_study)[T.2]    -0.2319      0.245     -0.948      0.343      -0.711       0.247
C(year_study)[T.3]    -0.0681      0.243     -0.281      0.779      -0.543       0.407
C(year_study)[T.4]    -0.2121      0.232     -0.913      0.361      -0.668       0.243
PU                     0.9979      0.090     11.036      0.000       0.821       1.175
PS                    -0.0669      0.092     -0.723      0.470      -0.248       0.114
EOU                    0.5077      0.106      4.794      0.000       0.300       0.715
finlit                -0.0101      0.087     -0.116      0.908      -0.181       0.161
female                -0.0563      0.185     -0.304      0.761      -0.419       0.306
ln_allow               0.8904      0.197   

## RQ3: ordinal outcome — ordered logit
$intention^*_i = x_i'\beta + \varepsilon_i$, and $intention_i = j$ if $\kappa_{j-1} < intention^*_i \le \kappa_j$

In [5]:
d3 = df.dropna(subset=["intention", "PU", "PS", "EOU", "finlit", "female", "ln_allow", "urban"])
ologit = OrderedModel(d3["intention"].astype(int), d3[["PU", "PS", "EOU", "finlit", "female", "ln_allow", "urban"]],
                      distr="logit").fit(method="bfgs", disp=False)
print(ologit.params.round(3))

PU          1.563
PS          0.923
EOU         0.705
finlit      0.006
female      0.041
ln_allow   -0.408
urban       0.008
1/2         4.484
2/3         0.587
3/4         0.422
4/5         0.550
dtype: float64

In [6]:
assert int(logit.nobs) == 394
assert abs(logit.params["PU"] - 0.874) < 0.001 and abs(lpm.params["PU"] - 0.128) < 0.001
print("Logit and LPM coefficients match the Stata do-file.")

Logit and LPM coefficients match the Stata do-file.
